# FT7B — RESUME ONLY after Kaggle 12h timeout

**KHÔNG dùng để train từ đầu**. Attach checkpoint Output của run FT7B đã timeout.

Notebook tìm checkpoint đầy đủ (`trainer_state.json`, optimizer, scheduler, weights),
giữ nguyên split/hyperparameters, copy best adapter từ epoch trước,
và chỉ train phần còn lại đến epoch 3. Nếu không tìm thấy checkpoint phù hợp,
dừng sớm để tránh vô tình train lại từ đầu.

# CODe Qwen2.5-VL-7B — LoRA FINETUNE ONLY

Notebook độc lập: **không chạy zero-shot**.

- 3 epochs
- cutoff 4096
- LR 5e-4
- cosine
- warmup 0.1
- LoRA all-linear
- checkpoint mỗi 100 optimizer steps
- nếu notebook fail / hết thời gian: attach output notebook fail vào lần chạy mới, notebook tự tìm checkpoint mới nhất và resume.
- `torchao` bị gỡ vì không dùng quantization và bản Kaggle 0.10 xung đột PEFT 0.21.

**v2 fix:** Added `inspect` and robust `TrainingArguments` API compatibility.


In [6]:
from pathlib import Path
import inspect
import os, sys, subprocess, json, ast, re, time, random, shutil, math
import importlib, importlib.metadata as mdlib

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SEED = 42
random.seed(SEED)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("WARNING: nvidia-smi not found.")


Mon Oct 12 07:11:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
def find_full_asset():
    candidates = []
    for p in INPUT.rglob("cls_train.json"):
        root = p.parent.parent
        if (
            (root / "data" / "cls_test.csv").exists()
            and (root / "wheelhouse").exists()
        ):
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy code_full_offline_assets. "
            "Hãy Add Input output/dataset từ notebook 00 FULL."
        )
    return candidates[0]

ASSET = find_full_asset()
DATA_DIR = ASSET / "data"
WHEEL_DIR = ASSET / "wheelhouse"
TRAIN_JSON = DATA_DIR / "cls_train.json"
TEST_CSV = DATA_DIR / "cls_test.csv"

print("ASSET:", ASSET)
print("TRAIN:", TRAIN_JSON)
print("TEST :", TEST_CSV)
print("WHEELS:", WHEEL_DIR)

ASSET: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets
TRAIN: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/data/cls_train.json
TEST : /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/data/cls_test.csv
WHEELS: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse


In [8]:
# RESUME-ONLY: find a COMPLETE optimizer checkpoint from a previous FT7B Output.
# The previous notebook hit Kaggle's runtime limit; do not restart from step 0.
# Required: Kaggle Add Input -> prior timed-out FT7B notebook Output.

from pathlib import Path
import json

prior_runs = sorted({p for p in INPUT.rglob("stage_ft7b") if p.is_dir()})
print("Prior FT7B stage folders:", len(prior_runs))
for folder in prior_runs:
    print(" -", folder)
    print("   checkpoints:", [p.name for p in folder.glob("checkpoint-*")])
    print("   best_adapter:", (folder / "best_adapter" / "adapter_config.json").exists())

valid = []
invalid = []
for stage_dir in prior_runs:
    for cp in stage_dir.glob("checkpoint-*"):
        if not cp.is_dir():
            continue
        try:
            state = json.loads((cp / "trainer_state.json").read_text(encoding="utf-8"))
            step = int(state.get("global_step", cp.name.split("-")[-1]))
        except Exception as exc:
            invalid.append((str(cp), "trainer_state.json missing or unreadable: " + repr(exc)))
            continue

        required = ["optimizer.pt", "scheduler.pt", "trainer_state.json"]
        missing = [f for f in required if not (cp / f).exists()]
        has_weights = any((cp / filename).exists() for filename in [
            "adapter_model.safetensors", "adapter_model.bin",
            "model.safetensors", "pytorch_model.bin",
        ])
        if not has_weights:
            missing.append("model or PEFT adapter weights")
        if missing:
            invalid.append((str(cp), "missing: " + ", ".join(missing)))
        else:
            valid.append((step, cp))

print("Complete checkpoint candidates:", len(valid))
for step, cp in sorted(valid):
    print(" step", step, "->", cp)
if invalid:
    print("Incomplete checkpoints (ignored):")
    for cp, reason in invalid:
        print(" ", cp, reason)

if not valid:
    bests = [p for p in INPUT.rglob("best_adapter/adapter_config.json") if "stage_ft7b" in str(p)]
    print("Best adapters found:", [str(p.parent) for p in bests])
    raise RuntimeError(
        "Không có checkpoint ĐẦY ĐỦ để resume optimizer. "
        "KHÔNG được chạy lại training từ đầu. "
        "Nếu có best_adapter, dùng notebook 41 đánh giá model epoch 2, "
        "hoặc gửi ảnh cây Output để quyết định cách tiếp tục adapter-only."
    )

PRIOR_STEP, PRIOR_CHECKPOINT = max(valid, key=lambda x: x[0])
PRIOR_STAGE = PRIOR_CHECKPOINT.parent
PRIOR_CFG = PRIOR_STAGE / "stage_config.json"
PRIOR_SPLIT = PRIOR_STAGE / "split.json"

print("✅ Selected prior checkpoint:", PRIOR_CHECKPOINT)
print("   optimizer step:", PRIOR_STEP)
print("   matching 3-epoch target: currently expected 1746 steps if previous logs are representative")

Prior FT7B stage folders: 1
 - /kaggle/input/notebooks/messi1231/40-7b-lora-finetune-only/stage_ft7b
   checkpoints: ['checkpoint-1500', 'checkpoint-1400', 'checkpoint-1600']
   best_adapter: True
Complete checkpoint candidates: 3
 step 1400 -> /kaggle/input/notebooks/messi1231/40-7b-lora-finetune-only/stage_ft7b/checkpoint-1400
 step 1500 -> /kaggle/input/notebooks/messi1231/40-7b-lora-finetune-only/stage_ft7b/checkpoint-1500
 step 1600 -> /kaggle/input/notebooks/messi1231/40-7b-lora-finetune-only/stage_ft7b/checkpoint-1600
✅ Selected prior checkpoint: /kaggle/input/notebooks/messi1231/40-7b-lora-finetune-only/stage_ft7b/checkpoint-1600
   optimizer step: 1600
   matching 3-epoch target: currently expected 1746 steps if previous logs are representative


In [9]:
# ---------- Offline environment setup ----------
# PEFT 0.21 checks optional torchao. Kaggle currently may ship torchao 0.10,
# which is too old for PEFT 0.21 and crashes LoRA injection.
# We do not use torchao quantization, so remove it BEFORE importing peft.
try:
    torchao_version = mdlib.version("torchao")
    print("Found torchao:", torchao_version, "-> uninstalling (not needed for this pipeline)")
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
        check=False, text=True
    )
except mdlib.PackageNotFoundError:
    print("torchao not installed: OK")

wheels = sorted(WHEEL_DIR.glob("*.whl"))

def wheel_dist_name(p):
    return p.name.split("-", 1)[0].replace("_", "-").lower()

wheel_map = {}
for w in wheels:
    wheel_map.setdefault(wheel_dist_name(w), []).append(w)

wanted = [
    "transformers", "peft", "accelerate", "qwen-vl-utils",
    "huggingface-hub", "safetensors", "tokenizers", "av"
]
selected = []
for name in wanted:
    xs = wheel_map.get(name, [])
    if xs:
        selected.append(sorted(xs)[-1])

print("Selective offline wheels:")
for w in selected:
    print(" -", w.name)

if selected:
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-index", "--no-deps", "--upgrade",
         *[str(w) for w in selected]],
        check=True
    )

# Verify in a fresh subprocess before expensive work.
preflight_code = r"""
import importlib.metadata as md
from PIL import Image
import torch, torchvision, transformers, peft, accelerate
from qwen_vl_utils import process_vision_info
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model, PeftModel
print("PREFLIGHT_OK")
print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
try:
    print("torchao", md.version("torchao"))
except md.PackageNotFoundError:
    print("torchao", "NOT_INSTALLED")
"""
pf = subprocess.run([sys.executable, "-c", preflight_code],
                    text=True, capture_output=True)
print(pf.stdout)
if pf.returncode != 0:
    print(pf.stderr)
    raise RuntimeError("Environment preflight failed; dừng trước khi load model.")

import numpy as np
import pandas as pd
import torch
import torchvision
import transformers
import peft
import accelerate
from PIL import Image
from transformers import (
    Qwen2_5_VLForConditionalGeneration, AutoProcessor,
    Trainer, TrainingArguments, TrainerCallback
)
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, get_peft_model, PeftModel

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for this notebook.")

ENV = {
    "python": sys.version,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "pillow": mdlib.version("Pillow"),
    "gpu": torch.cuda.get_device_name(0),
}
print(json.dumps(ENV, indent=2))

Found torchao: 0.10.0 -> uninstalling (not needed for this pipeline)
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Selective offline wheels:
 - transformers-4.57.6-py3-none-any.whl
 - peft-0.21.0-py3-none-any.whl
 - accelerate-1.15.0-py3-none-any.whl
 - qwen_vl_utils-0.0.14-py3-none-any.whl
 - huggingface_hub-0.36.2-py3-none-any.whl
 - safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 - tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 - av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse/transformers-4.57.6-py3-none-any.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse/peft-0.21.0-py3-none-any.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/w

In [10]:
train_data = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
test_df = pd.read_csv(TEST_CSV).fillna("")

def parse_images(x):
    if isinstance(x, (list, tuple)):
        return [str(v) for v in x]
    s = str(x).strip()
    if not s:
        return []
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)):
            return [str(v) for v in obj]
    except Exception:
        pass
    return [v.strip() for v in s.split(",") if v.strip()]

def local_path(rel):
    rel = str(rel).replace("\\", "/").lstrip("/")
    p = ASSET / rel
    if not p.exists():
        raise FileNotFoundError(f"Missing packaged image: {rel}")
    return p

IMAGE_TOKEN_LINE = re.compile(r"(?im)^\s*image-\d+\s*:\s*<image>\s*$")

def clean_image_placeholders(text):
    s = str(text)
    s = IMAGE_TOKEN_LINE.sub("", s)
    s = s.replace("<image>", "")
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict):
                if "text" in part:
                    parts.append(str(part["text"]))
        return "\n".join(x for x in parts if x)
    return "" if content is None else str(content)

def extract_messages(item):
    user, assistant = "", ""

    msgs = item.get("messages")
    if isinstance(msgs, list):
        for m in msgs:
            role = str(m.get("role", m.get("from", ""))).lower()
            txt = content_to_text(m.get("content", m.get("value", "")))
            if role in {"user","human"} and not user:
                user = txt
            elif role in {"assistant","gpt"} and not assistant:
                assistant = txt

    conv = item.get("conversations")
    if isinstance(conv, list):
        for m in conv:
            role = str(m.get("from", m.get("role", ""))).lower()
            txt = content_to_text(m.get("value", m.get("content", "")))
            if role in {"user","human"} and not user:
                user = txt
            elif role in {"assistant","gpt"} and not assistant:
                assistant = txt

    if not user:
        user = item.get("question", item.get("q", item.get("prompt", "")))
    if not assistant:
        assistant = item.get("answer", item.get("a", item.get("label", "")))

    image_field = item.get(
        "images",
        item.get("image", item.get("photos", item.get("photographs", [])))
    )
    images = [local_path(x) for x in parse_images(image_field)]
    return clean_image_placeholders(user), content_to_text(assistant).strip(), images

lower = {str(c).lower(): c for c in test_df.columns}
TEST_IMAGES_COL = next((lower[x] for x in ["images","image","photos","photographs"] if x in lower), None)
TEST_Q_COL = next((lower[x] for x in ["question","q","prompt","user"] if x in lower), None)
TEST_A_COL = next((lower[x] for x in ["answer","a","label","assistant"] if x in lower), None)
TEST_ID_COL = next((lower[x] for x in ["id","checkup_id","patient_id"] if x in lower), None)

if TEST_IMAGES_COL is None or TEST_Q_COL is None or TEST_A_COL is None:
    raise RuntimeError("Test CSV thiếu images/question/answer aliases.")

print("Train samples:", len(train_data))
print("Test samples :", len(test_df))

Train samples: 14696
Test samples : 1793


In [11]:
def find_7b_model():
    for p in INPUT.rglob("Qwen2.5-VL-7B-Instruct"):
        if p.is_dir() and (p / "config.json").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Qwen2.5-VL-7B-Instruct. "
        "Hãy Add Input output của notebook 30_prepare_7B_model_ONLINE."
    )

MODEL_DIR = find_7b_model()
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor = AutoProcessor.from_pretrained(str(MODEL_DIR), local_files_only=True)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR), torch_dtype=dtype,
    low_cpu_mem_usage=True, local_files_only=True
).to("cuda")
model.eval()
print("Loaded 7B:", MODEL_DIR)

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Loaded 7B: /kaggle/input/datasets/messi1231/prepare-7b/qwen7b_asset/Qwen2.5-VL-7B-Instruct


In [12]:
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class RawCODeDataset(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

class QwenVLTrainCollator:
    def __init__(self, processor, max_length=4096):
        self.processor = processor
        self.max_length = max_length

    def _encode_one(self, item):
        user, assistant, image_paths = extract_messages(item)
        user_content = [{"type":"image","image":str(p)} for p in image_paths]
        user_content.append({"type":"text","text":user})
        prompt_messages = [{"role":"user","content":user_content}]
        full_messages = prompt_messages + [{
            "role":"assistant",
            "content":[{"type":"text","text":assistant}]
        }]

        prompt_text = self.processor.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
        full_text = self.processor.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        )
        image_inputs, video_inputs = process_vision_info(full_messages)

        full = self.processor(
            text=[full_text], images=image_inputs, videos=video_inputs,
            padding=False, truncation=True, max_length=self.max_length,
            return_tensors="pt"
        )
        prompt = self.processor(
            text=[prompt_text], images=image_inputs, videos=video_inputs,
            padding=False, truncation=True, max_length=self.max_length,
            return_tensors="pt"
        )

        input_ids = full["input_ids"][0]
        labels = input_ids.clone()
        prompt_len = min(prompt["input_ids"].shape[1], labels.shape[0])
        labels[:prompt_len] = -100
        if torch.all(labels == -100):
            raise RuntimeError("Assistant answer fully truncated at cutoff_len=4096.")

        out = {
            "input_ids": input_ids,
            "attention_mask": full["attention_mask"][0],
            "labels": labels,
        }
        for key in ["pixel_values","image_grid_thw","pixel_values_videos","video_grid_thw"]:
            if key in full:
                out[key] = full[key]
        return out

    def __call__(self, features):
        encs = [self._encode_one(x) for x in features]
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = self.processor.tokenizer.eos_token_id
        batch = {
            "input_ids": pad_sequence([x["input_ids"] for x in encs], batch_first=True, padding_value=pad_id),
            "attention_mask": pad_sequence([x["attention_mask"] for x in encs], batch_first=True, padding_value=0),
            "labels": pad_sequence([x["labels"] for x in encs], batch_first=True, padding_value=-100),
        }
        for key in ["pixel_values","image_grid_thw","pixel_values_videos","video_grid_thw"]:
            vals = [x[key] for x in encs if key in x]
            if vals:
                batch[key] = torch.cat(vals, dim=0)
        return batch

In [13]:
STAGE = WORK / "stage_ft7b"
STAGE.mkdir(parents=True, exist_ok=True)
(STAGE / "environment.json").write_text(json.dumps(ENV, indent=2), encoding="utf-8")

# deterministic 95/5 split from released train only
idx = np.arange(len(train_data))
rng = np.random.default_rng(SEED)
rng.shuffle(idx)
n_val = max(1, int(round(0.05 * len(idx))))
val_idx = set(int(x) for x in idx[:n_val])
train_items = [x for i,x in enumerate(train_data) if i not in val_idx][:30000]
val_items = [x for i,x in enumerate(train_data) if i in val_idx]

split_meta = {
    "seed": SEED,
    "released_train": len(train_data),
    "train": len(train_items),
    "validation": len(val_items),
    "validation_indices": sorted(val_idx),
}
(STAGE / "split.json").write_text(json.dumps(split_meta, indent=2), encoding="utf-8")
print(split_meta)

processor.tokenizer.padding_side = "right"
collator = QwenVLTrainCollator(processor, max_length=4096)

# Fail fast on data before PEFT/training.
for n in [1, 2]:
    b = collator(train_items[:n])
    print("collator", n, {k: tuple(v.shape) for k,v in b.items()})
    del b
torch.cuda.empty_cache()

model.train()
model.config.use_cache = False
try:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
except TypeError:
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

lora_cfg = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
    target_modules="all-linear", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# Probe batch size; fallback while preserving approximate effective batch.
candidate_batches = list(range(3, 0, -1))
chosen_batch = None
chosen_grad_acc = None
probe_errors = {}

for bs in candidate_batches:
    try:
        probe = collator(train_items[:bs])
        probe = {k:v.to("cuda") for k,v in probe.items()}
        out = model(**probe)
        out.loss.backward()
        model.zero_grad(set_to_none=True)
        del out, probe
        torch.cuda.empty_cache()
        chosen_batch = bs
        chosen_grad_acc = math.ceil(24 / bs)
        print("Probe OK:", bs, "grad_acc:", chosen_grad_acc)
        break
    except torch.cuda.OutOfMemoryError as e:
        probe_errors[str(bs)] = repr(e)
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
        print("OOM probe batch", bs)

if chosen_batch is None:
    raise RuntimeError("Không có batch size nào từ probe chạy được.")

config = {
    "model":"Qwen2.5-VL-7B",
    "epochs":3,
    "cutoff_len":4096,
    "learning_rate":5e-4,
    "scheduler":"cosine",
    "warmup_ratio":0.1,
    "lora_r":8,
    "lora_alpha":16,
    "lora_target":"all-linear",
    "batch_size":chosen_batch,
    "gradient_accumulation":chosen_grad_acc,
    "target_effective_batch":24,
    "probe_errors":probe_errors,
}
(STAGE / "stage_config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

# Prevent accidental re-training from step 0 if previous output is missing.
resume = PRIOR_CHECKPOINT
if not (resume / "trainer_state.json").exists():
    raise RuntimeError("Resume checkpoint is no longer available.")

# Check that the exact train/validation split and effective batch settings are unchanged.
if PRIOR_SPLIT.exists():
    old_split = json.loads(PRIOR_SPLIT.read_text(encoding="utf-8"))
    old_ids = [int(v) for v in old_split.get("validation_indices", [])]
    if old_ids and old_ids != sorted(val_idx):
        raise RuntimeError("Old validation split differs; do not resume this run.")

if PRIOR_CFG.exists():
    old_cfg = json.loads(PRIOR_CFG.read_text(encoding="utf-8"))
    for key, cur in [
        ("batch_size", chosen_batch),
        ("gradient_accumulation", chosen_grad_acc),
        ("cutoff_len", 4096),
    ]:
        if key in old_cfg and old_cfg[key] != cur:
            raise RuntimeError(
                f"Checkpoint parameter mismatch: {key}: old={old_cfg[key]}, new={cur}. "
                "Do not silently resume with different optimizer scheduling."
            )

# Preserve the best validation adapter from the PREVIOUS Kaggle version.
# A fresh callback would otherwise reset best loss to +inf and overwrite the
# epoch-2 best adapter even if epoch 3 gets worse.
prior_best_adapter = PRIOR_STAGE / "best_adapter"
prior_best_meta = PRIOR_STAGE / "best_adapter_meta.json"
if (prior_best_adapter / "adapter_config.json").exists():
    dest_best_adapter = STAGE / "best_adapter"
    if dest_best_adapter.exists():
        shutil.rmtree(dest_best_adapter)
    shutil.copytree(prior_best_adapter, dest_best_adapter)
    if prior_best_meta.exists():
        shutil.copy2(prior_best_meta, STAGE / "best_adapter_meta.json")
    print("✅ Preserved previous best adapter:", dest_best_adapter)
else:
    print("WARNING: previous best_adapter absent. The optimizer checkpoint is intact;")
    print("         epoch-2 best adapter is not guaranteed preserved in new output.")

print("Resume checkpoint:", resume)

class BestAdapterCallback(TrainerCallback):
    def __init__(self, target_dir):
        self.target_dir = Path(target_dir)
        meta_path = self.target_dir.parent / "best_adapter_meta.json"
        self.best = float("inf")
        if meta_path.exists():
            try:
                prior_meta = json.loads(meta_path.read_text(encoding="utf-8"))
                self.best = float(prior_meta.get("eval_loss", float("inf")))
            except Exception as exc:
                print("Could not restore best validation loss:", exc)
        print("BestAdapterCallback initialized with best loss:", self.best)
    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        loss = (metrics or {}).get("eval_loss")
        if loss is not None and loss < self.best and model is not None:
            self.best = float(loss)
            if self.target_dir.exists():
                shutil.rmtree(self.target_dir)
            model.save_pretrained(self.target_dir)
            (self.target_dir.parent / "best_adapter_meta.json").write_text(
                json.dumps({"eval_loss":self.best,"step":state.global_step}, indent=2),
                encoding="utf-8"
            )
            print("Saved NEW BEST adapter:", self.best, "step", state.global_step)
        return control

# Build TrainingArguments dynamically for Transformers API compatibility.
ta_kwargs = dict(
    output_dir=str(STAGE),
    per_device_train_batch_size=chosen_batch,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=chosen_grad_acc,
    learning_rate=5e-4,
    num_train_epochs=3.0,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    report_to=[],
    seed=SEED,
    data_seed=SEED,
    optim="adamw_torch",
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
)

ta_sig = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in ta_sig:
    ta_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in ta_sig:
    ta_kwargs["evaluation_strategy"] = "epoch"
else:
    raise RuntimeError(
        "TrainingArguments hiện tại không hỗ trợ eval_strategy/evaluation_strategy."
    )

if "gradient_checkpointing_kwargs" in ta_sig:
    ta_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

args = TrainingArguments(**ta_kwargs)
print("TrainingArguments API OK")
print("Evaluation strategy:", getattr(args, "eval_strategy", getattr(args, "evaluation_strategy", None)))
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=RawCODeDataset(train_items),
    eval_dataset=RawCODeDataset(val_items),
    data_collator=collator,
    callbacks=[BestAdapterCallback(STAGE / "best_adapter")],
)

print("▶ RESUMING step", PRIOR_STEP, "from", resume)
result = trainer.train(resume_from_checkpoint=str(resume))

trainer.save_model(STAGE / "final_adapter")
processor.save_pretrained(STAGE / "processor")
trainer.save_state()

result_payload = {
    "metrics":result.metrics,
    "global_step":trainer.state.global_step,
    "best_adapter_meta": json.loads((STAGE/"best_adapter_meta.json").read_text())
        if (STAGE/"best_adapter_meta.json").exists() else None,
}
(STAGE / "train_result.json").write_text(
    json.dumps(result_payload, indent=2, default=str), encoding="utf-8"
)
print("✅ stage_ft7b training complete")

{'seed': 42, 'released_train': 14696, 'train': 13961, 'validation': 735, 'validation_indices': [16, 28, 74, 77, 105, 122, 143, 148, 170, 179, 182, 275, 285, 286, 303, 304, 342, 370, 420, 423, 501, 543, 557, 563, 570, 571, 624, 636, 642, 653, 722, 739, 742, 803, 844, 852, 862, 888, 895, 897, 900, 944, 956, 976, 990, 1018, 1028, 1032, 1059, 1063, 1071, 1075, 1136, 1162, 1170, 1183, 1196, 1199, 1218, 1239, 1260, 1282, 1353, 1356, 1364, 1370, 1395, 1402, 1406, 1411, 1413, 1421, 1431, 1443, 1452, 1539, 1567, 1586, 1589, 1590, 1593, 1615, 1641, 1650, 1653, 1664, 1686, 1690, 1704, 1723, 1737, 1739, 1789, 1838, 1844, 1888, 1916, 1939, 1940, 1962, 1965, 1979, 1992, 1997, 2014, 2032, 2036, 2041, 2060, 2068, 2083, 2107, 2122, 2128, 2202, 2286, 2303, 2353, 2358, 2367, 2373, 2375, 2405, 2444, 2451, 2461, 2469, 2480, 2507, 2516, 2519, 2580, 2628, 2643, 2675, 2711, 2714, 2761, 2816, 2822, 2850, 2855, 2908, 2969, 2978, 3064, 3089, 3130, 3157, 3177, 3180, 3197, 3204, 3206, 3242, 3250, 3261, 3278, 3299,

Epoch,Training Loss,Validation Loss
3,0.120100,0.150914


Saved NEW BEST adapter: 0.15091440081596375 step 1746
✅ stage_ft7b training complete
